# AI CRM Agent - Complete Backend

AI-powered CRM system with natural language querying, email campaigns, and audit logging.

In [45]:
import os
import json
import duckdb
import pandas as pd
import smtplib
import uuid
import hashlib
from datetime import datetime, timedelta
from typing import List, Dict, Optional
from dataclasses import dataclass, field, asdict
from pathlib import Path

from dotenv import load_dotenv
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

from langchain.tools import tool
from langchain.agents import create_agent
from langchain_core.prompts import ChatPromptTemplate

from langchain_groq import ChatGroq


In [46]:
import langchain.agents as agents
print(dir(agents))

['AgentState', '__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', 'create_agent', 'factory', 'middleware', 'structured_output']


In [47]:
import langchain
langchain.__version__

'1.3.2'

In [48]:
# ==========================================
# 1. CONFIGURATION & ENVIRONMENT SETUP
# ==========================================

load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
GMAIL_ADDRESS = os.getenv("GMAIL_ADDRESS")
GMAIL_APP_PASSWORD = os.getenv("GMAIL_APP_PASSWORD")

if not GROQ_API_KEY:
    raise ValueError("Missing GROQ_API_KEY in environment variables")

# Audit log file paths
AUDIT_LOG_PATH = Path("campaign_audit_log.jsonl")
SENT_EMAILS_LOG = Path("sent_emails_log.jsonl")

In [49]:
# ==========================================
# 2. DATA MODELS & STATE MANAGEMENT
# ==========================================

@dataclass
class Organization:
    name: str
    description: str
    created_at: str = field(default_factory=lambda: datetime.now().isoformat())

@dataclass
class EmailCampaign:
    campaign_id: str
    org_name: str
    query: str
    recipient_count: int
    subject: str
    body_template: str
    recipients: List[Dict]
    status: str = "draft"
    created_at: str = field(default_factory=lambda: datetime.now().isoformat())
    sent_emails: List[Dict] = field(default_factory=list)
    failed_emails: List[Dict] = field(default_factory=list)
    pending_emails: List[Dict] = field(default_factory=list)
    
    def to_dict(self):
        return asdict(self)

@dataclass
class EmailStatus:
    email: str
    status: str
    timestamp: str
    error: Optional[str] = None
    campaign_id: Optional[str] = None

In [50]:
# ==========================================
# 3. CRM DATABASE MANAGER
# ==========================================

class CRMDatabase:
    """
    Handles all database operations for customer data using DuckDB.
    Dynamically processes any Excel schema without fixed columns.
    """
    
    def __init__(self):
        self.conn = duckdb.connect(":memory:")
        self.df = None
        self.table_name = "customer_data"
        self.columns = []
        self.email_column = None
        self.name_column = None
        self._campaigns = {}
        self._conversation_history = []
        self._last_query_results = []
    
    def load_excel(self, file_path: str) -> Dict:
        """
        Load Excel file and register with DuckDB.
        Returns schema info and row count.
        """
        self.df = pd.read_excel(file_path)
        self.columns = list(self.df.columns)
        
        # Clean column names for SQL safety
        self.df.columns = [str(col).strip() for col in self.df.columns]
        
        # Detect email column
        for col in self.columns:
            col_lower = col.lower().replace(' ', '').replace('_', '')
            if 'email' in col_lower:
                self.email_column = col
                break
        
        # Detect name column
        for col in self.columns:
            col_lower = col.lower().replace(' ', '').replace('_', '')
            if any(name in col_lower for name in ['customername', 'clientname', 'name']):
                self.name_column = col
                break
        
        # Register with DuckDB
        self.conn.register(self.table_name, self.df)
        
        return {
            "row_count": len(self.df),
            "columns": self.columns,
            "email_column": self.email_column,
            "name_column": self.name_column,
            "sample": self.df.head(3).to_dict(orient='records')
        }
    
    def get_schema_and_sample(self) -> str:
        """
        Return schema and sample rows for LLM context.
        """
        if self.df is None:
            return "No data loaded. Please upload an Excel file first."
        
        schema_info = []
        for col in self.columns:
            dtype = str(self.df[col].dtype)
            schema_info.append(f"  {col}: {dtype}")
        
        sample = self.df.head(5).to_string()
        
        return f"""
TABLE: {self.table_name}
COLUMNS ({len(self.columns)}):
{chr(10).join(schema_info)}

SAMPLE DATA (5 rows):
{sample}

NOTE: Use exact column names with double quotes. Use ILIKE for text matching.
Date columns may need parsing with strptime or date functions.
"""
    
    def execute_safe_sql(self, sql: str) -> Dict:
        """
        Execute validated SELECT query.
        """
        if not self._validate_sql(sql):
            return {"error": "Unsafe SQL query detected. Only SELECT statements allowed."}
        
        try:
            result_df = self.conn.execute(sql).fetchdf()
            records = result_df.to_dict(orient='records')
            self._last_query_results = records
            return {
                "row_count": len(records),
                "columns": result_df.columns.tolist(),
                "data": records,
                "sql": sql
            }
        except Exception as e:
            return {"error": str(e), "sql": sql}
    
    def _validate_sql(self, sql: str) -> bool:
        """
        Validate SQL is safe (SELECT only).
        """
        sql_clean = sql.lower().strip()
        forbidden = ['drop', 'delete', 'update', 'insert', 'alter', 'truncate', 'create', 'replace', 'merge', 'grant', 'revoke']
        
        if not sql_clean.startswith('select'):
            return False
        
        for word in forbidden:
            if word in sql_clean:
                return False
        
        return True
    
    def get_customers_with_emails(self, records: List[Dict]) -> List[Dict]:
        """
        Filter records that have valid email addresses.
        """
        if not self.email_column:
            return []
        
        valid = []
        for record in records:
            email = record.get(self.email_column, '')
            if email and isinstance(email, str) and '@' in email and '.' in email:
                valid.append(record)
        return valid
    
    def export_results(self, format: str = 'csv') -> str:
        """
        Export last query results to file.
        """
        if not self._last_query_results:
            return "No results to export."
        
        export_df = pd.DataFrame(self._last_query_results)
        timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        filename = f"export_{timestamp}.{format}"
        
        if format == 'csv':
            export_df.to_csv(filename, index=False)
        elif format == 'excel':
            export_df.to_excel(filename, index=False)
        elif format == 'json':
            export_df.to_json(filename, orient='records', indent=2)
        
        return filename
    
    def add_conversation(self, role: str, content: str):
        self._conversation_history.append({"role": role, "content": content, "timestamp": datetime.now().isoformat()})
    
    def get_conversation_context(self) -> str:
        """
        Get recent conversation history for context.
        """
        recent = self._conversation_history[-5:]
        return json.dumps(recent, indent=2)

In [51]:
# ==========================================
# 4. AUDIT & LOGGING SYSTEM
# ==========================================

class AuditLogger:
    """
    Handles all audit logging for campaigns and email activities.
    Logs to JSONL files for easy parsing and append-only writes.
    """
    
    @staticmethod
    def log_campaign(campaign: EmailCampaign):
        entry = {
            "type": "campaign",
            "timestamp": datetime.now().isoformat(),
            "campaign_id": campaign.campaign_id,
            "org_name": campaign.org_name,
            "query": campaign.query,
            "recipient_count": campaign.recipient_count,
            "subject": campaign.subject,
            "status": campaign.status
        }
        with open(AUDIT_LOG_PATH, 'a') as f:
            f.write(json.dumps(entry) + '\n')
    
    @staticmethod
    def log_email_status(email_status: EmailStatus):
        entry = {
            "type": "email",
            "timestamp": email_status.timestamp,
            "email": email_status.email,
            "status": email_status.status,
            "error": email_status.error,
            "campaign_id": email_status.campaign_id
        }
        with open(SENT_EMAILS_LOG, 'a') as f:
            f.write(json.dumps(entry) + '\n')
    
    @staticmethod
    def get_audit_log(campaign_id: Optional[str] = None) -> List[Dict]:
        logs = []
        if AUDIT_LOG_PATH.exists():
            with open(AUDIT_LOG_PATH, 'r') as f:
                for line in f:
                    if line.strip():
                        entry = json.loads(line)
                        if campaign_id is None or entry.get('campaign_id') == campaign_id:
                            logs.append(entry)
        return logs
    
    @staticmethod
    def get_email_log(campaign_id: Optional[str] = None) -> List[Dict]:
        logs = []
        if SENT_EMAILS_LOG.exists():
            with open(SENT_EMAILS_LOG, 'r') as f:
                for line in f:
                    if line.strip():
                        entry = json.loads(line)
                        if campaign_id is None or entry.get('campaign_id') == campaign_id:
                            logs.append(entry)
        return logs

In [52]:
# ==========================================
# 5. EMAIL SERVICE
# ==========================================

class EmailService:
    """
    Handles email sending with tracking, duplicate prevention, and personalization.
    """
    
    def __init__(self, gmail_address: str, app_password: str):
        self.gmail_address = gmail_address
        self.app_password = app_password
        self._sent_hashes = set()
        self._load_sent_hashes()
    
    def _load_sent_hashes(self):
        """
        Load previously sent email hashes to prevent duplicates.
        """
        if SENT_EMAILS_LOG.exists():
            with open(SENT_EMAILS_LOG, 'r') as f:
                for line in f:
                    if line.strip():
                        entry = json.loads(line)
                        if entry.get('status') == 'sent':
                            email_hash = self._generate_hash(entry.get('email', ''), entry.get('campaign_id', ''))
                            self._sent_hashes.add(email_hash)
    
    def _generate_hash(self, email: str, campaign_id: str) -> str:
        return hashlib.sha256(f"{email}:{campaign_id}".encode()).hexdigest()
    
    def _is_duplicate(self, email: str, campaign_id: str) -> bool:
        return self._generate_hash(email, campaign_id) in self._sent_hashes
    
    def send_single_email(self, recipient: str, subject: str, body: str, campaign_id: str, customer_data: Optional[Dict] = None) -> EmailStatus:
        """
        Send a single email with tracking and duplicate prevention.
        """
        if self._is_duplicate(recipient, campaign_id):
            return EmailStatus(
                email=recipient,
                status="duplicate",
                timestamp=datetime.now().isoformat(),
                error="Email already sent in this campaign",
                campaign_id=campaign_id
            )
        
        try:
            msg = MIMEMultipart('alternative')
            msg['Subject'] = subject
            msg['From'] = self.gmail_address
            msg['To'] = recipient
            
            # Create HTML version
            html_body = body.replace('\n', '<br>')
            html_content = f"""
            <html>
            <body style="font-family: Arial, sans-serif; line-height: 1.6; color: #333;">
                <div style="max-width: 600px; margin: 0 auto; padding: 20px;">
                    {html_body}
                    <hr style="margin-top: 30px; border: none; border-top: 1px solid #eee;">
                    <p style="font-size: 12px; color: #666;">
                        This email was sent via AI CRM Agent. Campaign ID: {campaign_id}
                    </p>
                </div>
            </body>
            </html>
            """
            
            msg.attach(MIMEText(body, 'plain'))
            msg.attach(MIMEText(html_content, 'html'))
            
            with smtplib.SMTP_SSL('smtp.gmail.com', 465) as server:
                server.login(self.gmail_address, self.app_password)
                server.send_message(msg)
            
            # Mark as sent
            self._sent_hashes.add(self._generate_hash(recipient, campaign_id))
            
            status = EmailStatus(
                email=recipient,
                status="sent",
                timestamp=datetime.now().isoformat(),
                campaign_id=campaign_id
            )
            AuditLogger.log_email_status(status)
            return status
            
        except Exception as e:
            status = EmailStatus(
                email=recipient,
                status="failed",
                timestamp=datetime.now().isoformat(),
                error=str(e),
                campaign_id=campaign_id
            )
            AuditLogger.log_email_status(status)
            return status
    
    def send_campaign_emails(self, campaign: EmailCampaign, db: 'CRMDatabase') -> Dict:
        """
        Send emails to all campaign recipients with tracking.
        """
        campaign.status = "sending"
        AuditLogger.log_campaign(campaign)
        
        sent = []
        failed = []
        duplicates = []
        
        for recipient_data in campaign.recipients:
            email = recipient_data.get(db.email_column, '') if db.email_column else ''
            if not email or '@' not in str(email):
                continue
            
            # Personalize email
            personalized_body = self._personalize_email(campaign.body_template, recipient_data)
            personalized_subject = self._personalize_email(campaign.subject, recipient_data)
            
            status = self.send_single_email(
                recipient=str(email),
                subject=personalized_subject,
                body=personalized_body,
                campaign_id=campaign.campaign_id,
                customer_data=recipient_data
            )
            
            if status.status == 'sent':
                sent.append(asdict(status))
            elif status.status == 'duplicate':
                duplicates.append(asdict(status))
            else:
                failed.append(asdict(status))
        
        campaign.sent_emails = sent
        campaign.failed_emails = failed
        campaign.status = "completed" if not failed else "partial"
        AuditLogger.log_campaign(campaign)
        
        return {
            "campaign_id": campaign.campaign_id,
            "total": len(campaign.recipients),
            "sent": len(sent),
            "failed": len(failed),
            "duplicates": len(duplicates),
            "status": campaign.status
        }
    
    def _personalize_email(self, template: str, customer_data: Dict) -> str:
        """
        Replace {{Column Name}} placeholders with customer data.
        """
        result = template
        for key, value in customer_data.items():
            placeholder = f"{{{key}}}"
            if placeholder in result:
                result = result.replace(placeholder, str(value) if pd.notna(value) else '')
        return result

In [53]:
# ==========================================
# 6. SQL GENERATION ENGINE
# ==========================================

class SQLGenerator:
    """
    Generates SQL queries from natural language using LLM.
    Includes conversation context for follow-up queries.
    """
    
    def __init__(self, llm: ChatGroq):
        self.llm = llm
        self.prompt_template = ChatPromptTemplate.from_template("""
You are an expert DuckDB SQL engineer.

Database table name: customer_data

Schema:
{schema}

Conversation History (for context):
{conversation_context}

User Question:
{question}

RULES:
1. Generate ONLY valid DuckDB SQL SELECT queries
2. Use ONLY table 'customer_data'
3. Use exact column names with double quotes (e.g., "Customer Name")
4. Use ILIKE for case-insensitive text matching
5. Only SELECT queries - no modifications
6. Use LIMIT 100 when appropriate
7. For dates: use strptime() or current_date for comparisons
8. For warranty expiry within N days: use date arithmetic with current_date
9. Return ALL columns with SELECT * unless user asks for specific ones
10. Handle multiple filters with AND/OR as needed
11. For location/city filtering: use ILIKE on the location column
12. For price/value comparisons: use standard operators (>, <, >=, <=)

Return only the SQL query, nothing else.
""")
    
    def generate(self, question: str, schema: str, conversation_context: str = '') -> str:
        chain = self.prompt_template | self.llm
        response = chain.invoke({
            "schema": schema,
            "question": question,
            "conversation_context": conversation_context
        })
        
        sql = response.content.strip()
        sql = sql.replace('```sql', '').replace('```', '').strip()
        return sql

In [54]:
# ==========================================
# 7. EMAIL GENERATION ENGINE
# ==========================================

class EmailGenerator:
    """
    Generates personalized email campaigns using LLM.
    """
    
    def __init__(self, llm: ChatGroq, org_name: str, org_description: str):
        self.llm = llm
        self.org_name = org_name
        self.org_description = org_description
    
    def generate_campaign(self, query: str, recipient_count: int, sample_data: List[Dict], campaign_prompt: str) -> Dict:
        """
        Generate email subject and body for a campaign.
        """
        sample_json = json.dumps(sample_data[:3], indent=2, default=str)
        
        prompt = f"""
You are a professional marketing copywriter for {self.org_name}.

Company: {self.org_name}
Description: {self.org_description}

Campaign Context:
- Target audience: {recipient_count} customers
- Original query: {query}
- Sample customer data: {sample_json}

User's campaign instruction:
{campaign_prompt}

Generate a professional email campaign with:
1. Subject line (compelling, under 60 characters)
2. Email body (professional, personalized, clear CTA)

Use {{Column Name}} placeholders for personalization (e.g., {{Customer Name}}, {{Product}}).
Keep the tone professional and friendly.
Include a clear call-to-action.

Return ONLY a JSON object with 'subject' and 'body' keys.
"""
        
        response = self.llm.invoke(prompt)
        content = response.content.strip()
        content = content.replace('```json', '').replace('```', '').strip()
        
        try:
            return json.loads(content)
        except json.JSONDecodeError:
            # Fallback parsing
            return {
                "subject": f"Important Update from {self.org_name}",
                "body": content
            }
    
    def preview_email(self, body_template: str, subject_template: str, customer_data: Dict) -> Dict:
        """
        Generate preview of email for a specific customer by replacing placeholders.
        """
        result = {"subject": subject_template, "body": body_template}
        for key, value in customer_data.items():
            placeholder = f"{{{key}}}"
            if placeholder in result['subject']:
                result['subject'] = result['subject'].replace(placeholder, str(value) if pd.notna(value) else '')
            if placeholder in result['body']:
                result['body'] = result['body'].replace(placeholder, str(value) if pd.notna(value) else '')
        return result

In [55]:
# ==========================================
# 8. LANGCHAIN TOOLS
# ==========================================

# Global instances (initialized after setup)
db = None
sql_generator = None
email_generator = None
email_service = None
org = None
llm = None

@tool
def get_schema_and_sample() -> str:
    """
    Return schema and sample rows from the database.
    Use this to understand the data structure before querying.
    """
    if db is None:
        return "Database not initialized. Please upload data first."
    return db.get_schema_and_sample()

@tool
def query_customers(question: str) -> str:
    """
    Answer questions using customer database.
    Generates SQL from natural language and executes it safely.
    Returns complete customer records with all columns.
    """
    if db is None or sql_generator is None:
        return json.dumps({"error": "System not initialized. Please complete setup first."})
    
    schema = db.get_schema_and_sample()
    conversation_context = db.get_conversation_context()
    
    sql = sql_generator.generate(question, schema, conversation_context)
    print(f"\nGenerated SQL: {sql}")
    
    result = db.execute_safe_sql(sql)
    
    db.add_conversation("user", question)
    db.add_conversation("assistant", json.dumps(result, default=str))
    
    return json.dumps(result, default=str, indent=2)

@tool
def preview_campaign(recipients_json: str, subject: str, body: str) -> str:
    """
    Preview campaign with recipients, subject, and body.
    Creates a campaign in pending_approval status.
    """
    try:
        recipients = json.loads(recipients_json)
    except:
        return json.dumps({"error": "Invalid recipients JSON"})
    
    campaign_id = str(uuid.uuid4())[:8]
    campaign = EmailCampaign(
        campaign_id=campaign_id,
        org_name=org.name if org else "Unknown",
        query="manual",
        recipient_count=len(recipients),
        subject=subject,
        body_template=body,
        recipients=recipients,
        status="pending_approval"
    )
    
    db._campaigns[campaign_id] = campaign
    AuditLogger.log_campaign(campaign)
    
    return json.dumps({
        "campaign_id": campaign_id,
        "recipient_count": len(recipients),
        "sample": recipients[:5],
        "subject": subject,
        "body": body,
        "status": "pending_approval"
    }, indent=2, default=str)

@tool
def send_email(recipient: str, subject: str, body: str) -> str:
    """
    Send a single email (for testing or individual sends).
    """
    if email_service is None:
        return "Email service not configured."
    
    status = email_service.send_single_email(
        recipient=recipient,
        subject=subject,
        body=body,
        campaign_id="manual_" + str(uuid.uuid4())[:8]
    )
    return json.dumps(asdict(status), indent=2)

@tool
def generate_email_campaign(query: str, campaign_prompt: str, recipients_json: str) -> str:
    """
    Generate email campaign for selected customers using AI.
    Creates subject and body with personalization placeholders.
    """
    try:
        recipients = json.loads(recipients_json)
    except:
        return json.dumps({"error": "Invalid recipients JSON"})
    
    if not recipients:
        return json.dumps({"error": "No recipients provided"})
    
    campaign_data = email_generator.generate_campaign(
        query=query,
        recipient_count=len(recipients),
        sample_data=recipients[:3],
        campaign_prompt=campaign_prompt
    )
    
    campaign_id = str(uuid.uuid4())[:8]
    campaign = EmailCampaign(
        campaign_id=campaign_id,
        org_name=org.name if org else "Unknown",
        query=query,
        recipient_count=len(recipients),
        subject=campaign_data['subject'],
        body_template=campaign_data['body'],
        recipients=recipients,
        status="pending_approval"
    )
    
    db._campaigns[campaign_id] = campaign
    AuditLogger.log_campaign(campaign)
    
    return json.dumps({
        "campaign_id": campaign_id,
        "subject": campaign_data['subject'],
        "body_template": campaign_data['body'],
        "recipient_count": len(recipients),
        "status": "pending_approval"
    }, indent=2)

@tool
def approve_and_send_campaign(campaign_id: str) -> str:
    """
    Approve and send a pending campaign.
    This sends emails to all recipients and tracks delivery status.
    """
    if campaign_id not in db._campaigns:
        return json.dumps({"error": f"Campaign {campaign_id} not found"})
    
    campaign = db._campaigns[campaign_id]
    
    if campaign.status != "pending_approval":
        return json.dumps({"error": f"Campaign status is {campaign.status}, cannot send"})
    
    result = email_service.send_campaign_emails(campaign, db)
    return json.dumps(result, indent=2)

@tool
def preview_customer_email(campaign_id: str, customer_index: int) -> str:
    """
    Preview email for a specific customer in a campaign.
    Shows personalized email with placeholders replaced.
    """
    if campaign_id not in db._campaigns:
        return json.dumps({"error": "Campaign not found"})
    
    campaign = db._campaigns[campaign_id]
    
    if customer_index >= len(campaign.recipients):
        return json.dumps({"error": "Invalid customer index"})
    
    customer = campaign.recipients[customer_index]
    preview = email_generator.preview_email(
        campaign.body_template,
        campaign.subject,
        customer
    )
    
    return json.dumps({
        "customer": customer,
        "subject": preview['subject'],
        "body": preview['body']
    }, indent=2, default=str)

@tool
def get_campaign_status(campaign_id: str) -> str:
    """
    Get full status of a campaign including sent/failed emails.
    """
    if campaign_id not in db._campaigns:
        return json.dumps({"error": "Campaign not found"})
    
    campaign = db._campaigns[campaign_id]
    return json.dumps(campaign.to_dict(), indent=2, default=str)

@tool
def export_query_results(format: str = 'csv') -> str:
    """
    Export last query results to file (csv, excel, or json).
    """
    if db is None:
        return "Database not initialized"
    filename = db.export_results(format)
    return json.dumps({"filename": filename, "format": format})

@tool
def get_audit_logs(campaign_id: Optional[str] = None) -> str:
    """
    Retrieve audit logs for campaigns or emails.
    Pass campaign_id to filter for a specific campaign.
    """
    campaign_logs = AuditLogger.get_audit_log(campaign_id)
    email_logs = AuditLogger.get_email_log(campaign_id)
    return json.dumps({
        "campaign_logs": campaign_logs,
        "email_logs": email_logs
    }, indent=2, default=str)

In [56]:
# ==========================================
# 9. SYSTEM PROMPT & AGENT SETUP
# ==========================================

def create_system_prompt(org_name: str, org_description: str) -> str:
    return f"""
You are an AI CRM Assistant for {org_name}.

Company Description: {org_description}

Your Roles:
- CRM Analyst
- Marketing Campaign Manager
- Customer Intelligence Agent

CAPABILITIES:
1. Query customer database using natural language
2. Analyze customer data and provide insights
3. Create and manage email campaigns
4. Generate personalized email content
5. Preview emails before sending
6. Send campaigns with delivery tracking
7. Export query results to files
8. Retrieve audit logs for compliance

WORKFLOW FOR QUERIES:
1. Use query_customers to get data
2. Summarize results clearly with total count
3. Show complete customer details, all columns
4. Maintain context for follow-up questions

WORKFLOW FOR CAMPAIGNS:
1. Identify recipients via query_customers
2. Explain the target audience
3. Use generate_email_campaign to create content
4. Present preview and campaign_id to user
5. Wait for explicit user approval (they will call approve_and_send_campaign)
6. NEVER send emails without explicit approval
7. After sending, provide delivery status report
IMPORTANT:
- Never use customer phone numbers as company contact information.
- Do not invent contact numbers.
- Use only the company contact information provided below.
Company Support Number: 1800-123-4567

RULES:
- Never invent data or columns
- Always use tools for database operations
- Never write SQL manually in responses
- Show complete customer details, not partial
- Maintain conversation context for follow-ups
- Respect customer data privacy
- Prevent duplicate email sending
- Log all activities for audit

When asked about customers, ALWAYS use query_customers tool.
When asked to send emails, ALWAYS create a campaign and wait for approval.
"""

TOOLS_AVAILABLE = [
    get_schema_and_sample,
    query_customers,
    preview_campaign,
    send_email,
    generate_email_campaign,
    approve_and_send_campaign,
    preview_customer_email,
    get_campaign_status,
    export_query_results,
    get_audit_logs
]

In [57]:
# ==========================================
# 10. MAIN CRM AGENT CLASS
# ==========================================

class CRMAgent:
    """
    Main orchestrator for the AI CRM system.
    Handles setup, querying, campaigns, and exports.
    """
    
    def __init__(self):
        self.db = CRMDatabase()
        self.email_service = None
        self.agent_executor = None
        self.org = None
        self.llm = None
        self.sql_generator = None
        self.email_generator = None
    
    def setup_organization(self, name: str, description: str) -> Dict:
        """
        Step 1: Configure organization details.
        """
        self.org = Organization(name=name, description=description)
        return {
            "status": "success",
            "org_name": name,
            "org_description": description
        }
    
    def load_data(self, file_path: str) -> Dict:
        """
        Step 2: Load Excel data into the system.
        Dynamically detects schema and email/name columns.
        """
        global db
        result = self.db.load_excel(file_path)
        db = self.db  # Set global for tools
        return result
    
    def initialize_llm(self, model: str = "llama-3.3-70b-versatile", temperature: float = 0.1) -> None:
        """
        Initialize LLM and dependent services.
        """
        global llm, sql_generator, email_generator, email_service, org
        
        self.llm = ChatGroq(
            model=model,
            temperature=temperature,
            groq_api_key=GROQ_API_KEY
        )
        llm = self.llm
        
        self.sql_generator = SQLGenerator(self.llm)
        sql_generator = self.sql_generator
        
        if self.org:
            self.email_generator = EmailGenerator(self.llm, self.org.name, self.org.description)
            email_generator = self.email_generator
            org = self.org
        
        if GMAIL_ADDRESS and GMAIL_APP_PASSWORD:
            self.email_service = EmailService(GMAIL_ADDRESS, GMAIL_APP_PASSWORD)
            email_service = self.email_service
    
    def create_agent(self) -> None:
        """
        Create the LangChain ReAct agent with all tools.
        """
        if self.llm is None:
            raise ValueError("LLM not initialized.")

        if self.org is None:
            raise ValueError("Organization not set up.")

        system_prompt = create_system_prompt(
            self.org.name,
            self.org.description
        )

        self.agent_executor = create_agent(
            model=self.llm,
            tools=TOOLS_AVAILABLE,
            system_prompt=system_prompt
        )
    
    def query(self, user_input: str) -> Dict:
        """
        Process natural language query through the agent.
        """
        if self.agent_executor is None:
            return {"error": "Agent not initialized. Complete setup first."}
        
        response = self.agent_executor.invoke({
    "messages": [
        {
            "role": "user",
            "content": user_input
        }
    ]
})
        
        return {
            "input": user_input,
            "output": response.get('output', ''),
            "intermediate_steps": response.get('intermediate_steps', [])
        }
    
    def get_last_results(self) -> List[Dict]:
        """
        Get results from last database query.
        """
        return self.db._last_query_results
    
    def get_campaign(self, campaign_id: str) -> Optional[EmailCampaign]:
        return self.db._campaigns.get(campaign_id)
    
    def get_all_campaigns(self) -> List[Dict]:
        return [c.to_dict() for c in self.db._campaigns.values()]

In [58]:
# ==========================================
# 11. INITIALIZATION & USAGE EXAMPLE
# ==========================================

# Initialize the CRM Agent
crm = CRMAgent()

# Step 1: Setup Organization
crm.setup_organization(
    name="Shetty Enterprises",
    description="We sell all types of electronic appliances and provide best quality services"
)

# Step 2: Initialize LLM
crm.initialize_llm()

# Step 3: Load Data (uncomment when you have the file)
# data_info = crm.load_data("sales.xlsx")
# print(json.dumps(data_info, indent=2))

# Step 4: Create Agent
crm.create_agent()

# Step 5: Query (uncomment after loading data)
# result = crm.query("Show me customers who bought Samsung phones")
# print(result['output'])

In [59]:
# ==========================================
# 12. DIRECT API FUNCTIONS (for frontend integration)
# ==========================================

def init_crm(org_name: str, org_description: str, excel_path: str, model: str = "llama-3.3-70b-versatile") -> CRMAgent:
    """
    Complete initialization in one call.
    Returns fully configured CRMAgent instance.
    """
    crm = CRMAgent()
    crm.setup_organization(org_name, org_description)
    crm.initialize_llm(model=model)
    crm.load_data(excel_path)
    crm.create_agent()
    return crm

def query_customers_direct(crm: CRMAgent, question: str) -> Dict:
    """
    Direct customer query without agent wrapper.
    Returns raw query results with all columns.
    """
    schema = crm.db.get_schema_and_sample()
    conversation_context = crm.db.get_conversation_context()
    sql = crm.sql_generator.generate(question, schema, conversation_context)
    result = crm.db.execute_safe_sql(sql)
    crm.db.add_conversation("user", question)
    crm.db.add_conversation("assistant", json.dumps(result, default=str))
    return result

def create_campaign_direct(crm: CRMAgent, query: str, campaign_prompt: str, recipients: List[Dict]) -> Dict:
    """
    Create campaign directly without agent.
    Returns campaign_id, subject, body_template.
    """
    campaign_data = crm.email_generator.generate_campaign(
        query=query,
        recipient_count=len(recipients),
        sample_data=recipients[:3],
        campaign_prompt=campaign_prompt
    )
    
    campaign_id = str(uuid.uuid4())[:8]
    campaign = EmailCampaign(
        campaign_id=campaign_id,
        org_name=crm.org.name,
        query=query,
        recipient_count=len(recipients),
        subject=campaign_data['subject'],
        body_template=campaign_data['body'],
        recipients=recipients,
        status="pending_approval"
    )
    
    crm.db._campaigns[campaign_id] = campaign
    AuditLogger.log_campaign(campaign)
    
    return {
        "campaign_id": campaign_id,
        "subject": campaign_data['subject'],
        "body_template": campaign_data['body'],
        "recipient_count": len(recipients),
        "status": "pending_approval"
    }

def preview_email_direct(crm: CRMAgent, campaign_id: str, customer_index: int = 0) -> Dict:
    """
    Preview email for specific customer.
    Shows personalized email with placeholders replaced.
    """
    campaign = crm.db._campaigns.get(campaign_id)
    if not campaign:
        return {"error": "Campaign not found"}
    
    customer = campaign.recipients[customer_index]
    preview = crm.email_generator.preview_email(
        campaign.body_template,
        campaign.subject,
        customer
    )
    return {
        "customer": customer,
        "subject": preview['subject'],
        "body": preview['body']
    }

def send_campaign_direct(crm: CRMAgent, campaign_id: str) -> Dict:
    """
    Send approved campaign.
    Returns delivery status report.
    """
    campaign = crm.db._campaigns.get(campaign_id)
    if not campaign:
        return {"error": "Campaign not found"}
    if campaign.status != "pending_approval":
        return {"error": f"Campaign status is {campaign.status}"}
    return crm.email_service.send_campaign_emails(campaign, crm.db)

def get_campaign_report(crm: CRMAgent, campaign_id: str) -> Dict:
    """
    Get full campaign report with audit logs.
    """
    campaign = crm.db._campaigns.get(campaign_id)
    if not campaign:
        return {"error": "Campaign not found"}
    
    return {
        "campaign": campaign.to_dict(),
        "audit_logs": AuditLogger.get_audit_log(campaign_id),
        "email_logs": AuditLogger.get_email_log(campaign_id)
    }

In [ ]:
# ==========================================
# 13. TEST / DEMONSTRATION FLOW
# ==========================================

# Example usage flow (run after setup and data loading):


crm = init_crm(
    org_name="Shetty Enterprises",
    org_description="We sell all types of electronic appliances and provide best quality services",
    excel_path="sales.xlsx"
)


results = query_customers_direct(crm, "Get all customers whose warranty expires within the next 30 days")
print(f"Found {results['row_count']} customers")
for r in results['data'][:3]:
    print(r)


campaign = create_campaign_direct(
    crm=crm,
    query="warranty expiring soon",
    campaign_prompt="Send an email informing customers that their warranty is about to expire and offer a discounted warranty extension plan.",
    recipients=results['data']
)
print(f"Campaign created: {campaign['campaign_id']}")


preview = preview_email_direct(crm, campaign['campaign_id'], 0)
print("Preview Subject:", preview['subject'])
print("Preview Body:", preview['body'][:200] + "...")




Found 3 customers
{'S.No': 1, 'Customer Name': 'Akshay Shetty', 'Phone': 6360011746, 'Email': 'akshayshetty747@gmail.com', 'Product': 'Iphone 17', 'Price': 100000, 'Qty': 1, 'Purchase date': Timestamp('2025-01-15 00:00:00'), 'Warranty End Date': Timestamp('2026-06-15 00:00:00')}
{'S.No': 2, 'Customer Name': 'Akshatha Shetty', 'Phone': 8904315911, 'Email': 'akshathashetty243@gmail.com', 'Product': 'Iphone 15', 'Price': 60000, 'Qty': 1, 'Purchase date': Timestamp('2025-01-25 00:00:00'), 'Warranty End Date': Timestamp('2026-06-25 00:00:00')}
{'S.No': 3, 'Customer Name': 'Akshay ', 'Phone': 8970042819, 'Email': '4mt22cs014@mite.ac.in', 'Product': 'Iphone 16', 'Price': 80000, 'Qty': 1, 'Purchase date': Timestamp('2025-02-18 00:00:00'), 'Warranty End Date': Timestamp('2026-06-18 00:00:00')}
Campaign created: 45e55c7a
Preview Subject: Warranty Expiring Soon
Preview Body: Dear Akshay Shetty,

We hope this email finds you enjoying your Iphone 17. As a valued customer of Shetty Enterprises, we a

In [ ]:
status = send_campaign_direct(crm, campaign['campaign_id'])
print(json.dumps(status, indent=2))


report = get_campaign_report(crm, campaign['campaign_id'])
print(json.dumps(report, indent=2, default=str))